In [1]:
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("dataset.csv")

df = df.drop(columns=["Unnamed: 0"], errors="ignore")

df = df.dropna(how="all")

print(df.head())

                 track_id                 artists  \
0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   
3  6lfxq3CG4xtTiEg7opyCyx            Kina Grannis   
4  5vjLSffimiIP26QG5WcN2K        Chord Overstreet   

                                          album_name  \
0                                             Comedy   
1                                   Ghost (Acoustic)   
2                                     To Begin Again   
3  Crazy Rich Asians (Original Motion Picture Sou...   
4                                            Hold On   

                   track_name  popularity  duration_ms  explicit  \
0                      Comedy          73       230666     False   
1            Ghost - Acoustic          55       149610     False   
2              To Begin Again          57       210826     False   
3  Can't Help Falling In Love          71       201933     False   
4   

In [3]:
df["Features"] = (
      df["track_name"].fillna("").astype(str)
    + " "
    + df["artists"].fillna("").astype(str)
    + " "
    + df["album_name"].fillna("").astype(str)
    + " "
    + df["explicit"].fillna("").astype(str)
)

print("\nCombined Features")
print("-" * 50)

print(df[["track_name", "Features"]].head())


Combined Features
--------------------------------------------------
                   track_name  \
0                      Comedy   
1            Ghost - Acoustic   
2              To Begin Again   
3  Can't Help Falling In Love   
4                     Hold On   

                                            Features  
0                    Comedy Gen Hoshino Comedy False  
1  Ghost - Acoustic Ben Woodward Ghost (Acoustic)...  
2  To Begin Again Ingrid Michaelson;ZAYN To Begin...  
3  Can't Help Falling In Love Kina Grannis Crazy ...  
4             Hold On Chord Overstreet Hold On False  


In [4]:
cv = CountVectorizer()

feature_matrix = cv.fit_transform(df["Features"])

print("\nFeature Matrix Shape")
print(feature_matrix.shape)

print(feature_matrix)


Feature Matrix Shape
(114000, 78057)
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 997845 stored elements and shape (114000, 78057)>
  Coords	Values
  (0, 13328)	2
  (0, 24529)	1
  (0, 28547)	1
  (0, 21508)	1
  (1, 21508)	1
  (1, 24826)	2
  (1, 1528)	2
  (1, 7185)	1
  (1, 69327)	1
  (2, 21508)	1
  (2, 63733)	2
  (2, 7013)	2
  (2, 1997)	2
  (2, 29971)	1
  (2, 40636)	1
  (2, 70593)	1
  (3, 21508)	1
  (3, 10496)	1
  (3, 27623)	1
  (3, 21496)	1
  (3, 29662)	1
  (3, 37293)	1
  (3, 33857)	1
  (3, 25694)	1
  (3, 14339)	1
  :	:
  (113996, 39914)	1
  (113996, 52523)	1
  (113996, 52507)	1
  (113996, 40940)	1
  (113996, 40939)	1
  (113997, 21508)	1
  (113997, 7424)	1
  (113997, 45340)	1
  (113997, 41151)	1
  (113997, 48027)	1
  (113997, 11545)	1
  (113997, 21002)	1
  (113998, 21508)	1
  (113998, 69366)	1
  (113998, 70285)	1
  (113998, 58574)	1
  (113998, 23389)	1
  (113998, 40633)	1
  (113998, 11750)	1
  (113999, 21508)	1
  (113999, 41151)	1
  (113999, 48027)	1
  (113999, 11545)	1


In [5]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

numeric_cols = [
    "popularity",
    "duration_ms",
    "danceability",
    "energy"
]

df[numeric_cols] = df[numeric_cols].apply(
    pd.to_numeric, errors="coerce"
)

df[numeric_cols] = df[numeric_cols].fillna(
    df[numeric_cols].median()
)

numeric_scaled = scaler.fit_transform(df[numeric_cols])

In [6]:
from scipy.sparse import csr_matrix

numeric_sparse = csr_matrix(numeric_scaled)

print(numeric_sparse)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 439821 stored elements and shape (114000, 4)>
  Coords	Values
  (0, 0)	0.73
  (0, 1)	0.04404296492750552
  (0, 2)	0.6862944162436548
  (0, 3)	0.461
  (1, 0)	0.55
  (1, 1)	0.028566273238379736
  (1, 2)	0.42639593908629436
  (1, 3)	0.166
  (2, 0)	0.5700000000000001
  (2, 1)	0.040254749827916894
  (2, 2)	0.4446700507614213
  (2, 3)	0.359
  (3, 0)	0.71
  (3, 1)	0.038556735872239394
  (3, 2)	0.2700507614213198
  (3, 3)	0.0596
  (4, 0)	0.8200000000000001
  (4, 1)	0.037968646028150026
  (4, 2)	0.6274111675126903
  (4, 3)	0.443
  (5, 0)	0.58
  (5, 1)	0.04090661305120296
  (5, 2)	0.6984771573604059
  (5, 3)	0.481
  (6, 0)	0.74
  :	:
  (113993, 3)	0.687
  (113994, 0)	0.22
  (113994, 1)	0.05832285559625723
  (113994, 2)	0.33604060913705586
  (113994, 3)	0.171
  (113995, 0)	0.21
  (113995, 1)	0.0735110395729093
  (113995, 2)	0.17461928934010149
  (113995, 3)	0.235
  (113996, 0)	0.22
  (113996, 1)	0.07351123051117037
  (113996, 2)	0.1766

In [7]:
from scipy.sparse import hstack

combined_matrix = hstack([
    feature_matrix,
    numeric_sparse
])

print(combined_matrix)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1437666 stored elements and shape (114000, 78061)>
  Coords	Values
  (0, 13328)	2.0
  (0, 24529)	1.0
  (0, 28547)	1.0
  (0, 21508)	1.0
  (0, 78057)	0.73
  (0, 78058)	0.04404296492750552
  (0, 78059)	0.6862944162436548
  (0, 78060)	0.461
  (1, 21508)	1.0
  (1, 24826)	2.0
  (1, 1528)	2.0
  (1, 7185)	1.0
  (1, 69327)	1.0
  (1, 78057)	0.55
  (1, 78058)	0.028566273238379736
  (1, 78059)	0.42639593908629436
  (1, 78060)	0.166
  (2, 21508)	1.0
  (2, 63733)	2.0
  (2, 7013)	2.0
  (2, 1997)	2.0
  (2, 29971)	1.0
  (2, 40636)	1.0
  (2, 70593)	1.0
  (2, 78057)	0.5700000000000001
  :	:
  (113997, 78057)	0.22
  (113997, 78058)	0.051833245979078894
  (113997, 78059)	0.6385786802030456
  (113997, 78060)	0.329
  (113998, 21508)	1.0
  (113998, 69366)	1.0
  (113998, 70285)	1.0
  (113998, 58574)	1.0
  (113998, 23389)	1.0
  (113998, 40633)	1.0
  (113998, 11750)	1.0
  (113998, 78057)	0.41000000000000003
  (113998, 78058)	0.05420603574937062
  (113

In [8]:
print("\nCombined Feature Matrix")
print("-" * 50)

print(combined_matrix)


Combined Feature Matrix
--------------------------------------------------
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1437666 stored elements and shape (114000, 78061)>
  Coords	Values
  (0, 13328)	2.0
  (0, 24529)	1.0
  (0, 28547)	1.0
  (0, 21508)	1.0
  (0, 78057)	0.73
  (0, 78058)	0.04404296492750552
  (0, 78059)	0.6862944162436548
  (0, 78060)	0.461
  (1, 21508)	1.0
  (1, 24826)	2.0
  (1, 1528)	2.0
  (1, 7185)	1.0
  (1, 69327)	1.0
  (1, 78057)	0.55
  (1, 78058)	0.028566273238379736
  (1, 78059)	0.42639593908629436
  (1, 78060)	0.166
  (2, 21508)	1.0
  (2, 63733)	2.0
  (2, 7013)	2.0
  (2, 1997)	2.0
  (2, 29971)	1.0
  (2, 40636)	1.0
  (2, 70593)	1.0
  (2, 78057)	0.5700000000000001
  :	:
  (113997, 78057)	0.22
  (113997, 78058)	0.051833245979078894
  (113997, 78059)	0.6385786802030456
  (113997, 78060)	0.329
  (113998, 21508)	1.0
  (113998, 69366)	1.0
  (113998, 70285)	1.0
  (113998, 58574)	1.0
  (113998, 23389)	1.0
  (113998, 40633)	1.0
  (113998, 11750)	1.0
  (113

In [10]:
similarity = cosine_similarity(feature_matrix)

print("\nSimilarity Matrix")
print("-"*50)
print(similarity)

MemoryError: Unable to allocate 96.8 GiB for an array with shape (114000, 114000) and data type float64